# 8-PUZZLE DATASET

Generates 50000 random games using the project's standard formulation, solves each game with ASTS, and saves a CSV file containing the flattened configurations and the selected move encoded as one-hot:

1. nine numerical attributes, with values from 1 to 8 in the flattened positions and 0 representing the empty space;
2. four boolean labels indicating the selected move, corresponding to the variables up, right, down, and left, in that order.

## SETTING

Importing the libraries:

In [1]:
import sys
import csv
import types

from pathlib import Path

Find the project root:

In [2]:
def find_project_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()

    for path in (start, *start.parents):
        if (path / 'puzzle').is_dir():
            return path

    raise RuntimeError('Could not locate the project root.')


PROJECT_ROOT  = find_project_root()
NOTEBOOKS_DIR = PROJECT_ROOT  / 'notebooks'
DATA_DIR      = NOTEBOOKS_DIR / 'data'

DATA_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

Importing the puzzle and solver components:

In [3]:
from puzzle.search_algorithm import ASTS
from puzzle                  import formulations

pygame 2.5.2 (SDL 2.28.2, Python 3.12.12)
Hello from the pygame community. https://www.pygame.org/contribute.html


## PARAMETERS

In [4]:
N         = 3
NUM_GAMES = 50000

OUTPUT_PATH = DATA_DIR / '8_puzzle_asts.csv'

In [5]:
POSITION_COLUMNS = [
    'position' + str(idx)
    for idx in range(N**2)
]
MOVE_COLUMNS = ['move_up'   ,
                'move_right',
                'move_down' ,
                'move_left' ]

COLUMNS = POSITION_COLUMNS + MOVE_COLUMNS

ACTION_TO_ONE_HOT = {
    'u' : [1, 0, 0, 0],
    'r' : [0, 1, 0, 0],
    'd' : [0, 0, 1, 0],
    'l' : [0, 0, 0, 1],
}

## GAME GENERATION AND SOLVER HELPERS

In [6]:
def make_random_games(num_games=NUM_GAMES, n=N):
    return [
        formulations.new_grid(n)
        for _ in range(num_games)
    ]


def solve_game(grid):
    solver = ASTS(grid)

    solver.search()

    if solver.solution is None:
        raise RuntimeError(f'ASTS could not find a solution for:\n{grid}')

    return solver.solution


def solution_to_rows(solution):
    rows = []

    for current_node, next_node in zip(solution[:-1], solution[1:]):
        positions = current_node.state.flatten().astype(int).tolist()
        movement  = ACTION_TO_ONE_HOT[next_node.action]

        rows.append(positions + movement)

    return rows

## APPLY

Generating the new games:

In [7]:
games = make_random_games()

Solving the games and building the dataset rows:

In [8]:
dataset_rows = []

for game_idx, grid in enumerate(games, start=1):
    solution = solve_game(grid)

    dataset_rows.extend(solution_to_rows(solution))

    if game_idx % 1500 == 0 or game_idx == NUM_GAMES:
        print(f'{game_idx}/{NUM_GAMES} games solved; {len(dataset_rows)} moves collected.')

1500/50000 games solved; 22735 moves collected.
3000/50000 games solved; 45295 moves collected.
4500/50000 games solved; 67632 moves collected.
6000/50000 games solved; 90006 moves collected.
7500/50000 games solved; 112550 moves collected.
9000/50000 games solved; 135086 moves collected.
10500/50000 games solved; 157653 moves collected.
12000/50000 games solved; 179691 moves collected.
13500/50000 games solved; 202032 moves collected.
15000/50000 games solved; 224396 moves collected.
16500/50000 games solved; 246625 moves collected.
18000/50000 games solved; 268798 moves collected.
19500/50000 games solved; 291187 moves collected.
21000/50000 games solved; 313610 moves collected.
22500/50000 games solved; 335976 moves collected.
24000/50000 games solved; 358450 moves collected.
25500/50000 games solved; 380971 moves collected.
27000/50000 games solved; 403400 moves collected.
28500/50000 games solved; 425957 moves collected.
30000/50000 games solved; 447957 moves collected.
31500/5000

In [9]:
print(f'Dataset rows = {len(dataset_rows)}')
print(f'Columns      = {len(COLUMNS     )}')

Dataset rows = 746914
Columns      = 13


## SAVE

In [10]:
dataset_rows = list(dict.fromkeys(tuple(row) for row in dataset_rows))

with OUTPUT_PATH.open('w', newline='') as csv_file:
    writer = csv.writer(csv_file)

    writer.writerow (COLUMNS     )
    writer.writerows(dataset_rows)


print(f'Dataset saved to: {OUTPUT_PATH}!')

Dataset saved to: /home/rei-luisinho/solving_8_puzzle/notebooks/data/8_puzzle_asts.csv!
